## GPQA-D

#### Experiments:
- Testees : gpt-4.1-mini, qwen2.5-7b-it
- Judges : qwen3-4b
1) Baseline - free text
2) Baseline-wrong - all deliberate wrong answers
3) Multiple Answers
    - A. Forward
    - B. Backward
4) Surface Manipulation

In [18]:
from metrics_gpqa import mean_accuracy, calc_asr, decision_flip, significance, cohens
from tabulate import tabulate
import pandas as pd
import textwrap
import plotly.express as px


def wrap_text(s, width=11):
    if isinstance(s, str):
        return "\n".join(textwrap.wrap(s, width=width))
    return s
def get_exp(path):
    if "gpt" in path.lower():
        testee = "gpt-4.1-mini"
    if "qwen" in path.lower():
        testee = "qwen2.5-7B-it"
    
    if "qual" in path.lower():
        qtype = "Qualitative"
    if "quant" in path.lower():
        qtype = "Quantitative"

    if "wrong" in path.lower():
        exp = "baseline-incorrect"
        
    elif "forward" in path.lower():
        exp = "multiple-forward"
    elif "backward" in path.lower():
        exp = "multiple-backward"
    elif "strategic" in path.lower():
        exp = "strategic"
    else:
        exp = "baseline"
    return testee, exp, qtype



In [19]:
def get_basic_metrics(scores):
    results = []
    judge = "qwen3-4B"
    for path in scores:
        
        testee, exp, qtype = get_exp(path)

        df = pd.read_csv(path)
        acc, num_samples = mean_accuracy(df)

        base_asr_qual, total_base_qual = calc_asr(path)

        # print(num_samples == total_base_qual)
        results.append([judge, testee, exp, qtype, num_samples, acc])
    headers = ["Judge", "Testee", "Experiment", "Q-Type", "Data Samples", "Accuracy"]
    table = tabulate(results, headers=headers, tablefmt="grid")
    print(table)
    return results


In [20]:
#baseline, attack pair paths to scores

def get_comparisons(comparisons):
    results = []
    judge = "qwen3-4B"
    for base, game in comparisons:
        if "qual" in base.lower():
            qtype = "Qualitative"
        if "quant" in base.lower():
            qtype = "Quantitative"
        if "gpt" in base.lower():
            testee = "gpt-4.1-mini"
        if "qwen" in base.lower():
            testee = "qwen2.5-7B-it"
        exp = ""
        for path in [base, game]:
            if "wrong" in path.lower():
                exp += "baseline-incorrect/"
            elif "forward" in path.lower():
                exp += "multiple-forward/"
            elif "backward" in path.lower():
                exp += "multiple-backward/"
            elif "strategic" in path.lower():
                exp += "strategic/"
            else:
                exp += "baseline/"

        # bdf = pd.read_csv(base)
        # acc, num_samples = mean_accuracy(bdf)

        base_asr, total_base = calc_asr(base)
        game_asr, total_game = calc_asr(game)
        base_suc = base_asr*total_base
        game_successes = game_asr*total_game

        flips_asr, flip_successes, game_samples = decision_flip(base, game)
        
        #base successes, base_total_num, game_successes, game_total_num
        base_asr, attack_asr, zstat, pval = significance(base_suc, total_base, game_successes, game_samples)
        
        coh = cohens(base, game)
        cohensd = coh['cohen_d']
        cohensh = coh['cohen_h']

        results.append([judge, testee, exp, qtype, flip_successes, flips_asr, pval, cohensd, base_asr, attack_asr, total_base, game_samples, base_suc, game_successes,   ])
        

    wrapped = [[wrap_text(cell) for cell in row] for row in results]
    headers = ["Judge", "Testee", "Experiment", "Q-Type","Decision Flips", "Decision Flip %", "p-value", "Cohen's d","Base ASR", "Gamed ASR", "Base Samples", "Gamed Samples", "Base Successes", "Gamed Successes",]
    table = tabulate(results, headers=headers, tablefmt="grid")
    print(table)
    return results



In [28]:
exprs = ["baseline", "strategic", "forward"]
btypes = ["qual", "quant"]
testees = ["gpt", "qwen"]
bench = "gpqa"
scores = []
for expr in exprs:
    for testee in testees:
        for btype in btypes:
        
            scores.append(f"scores/{bench}/{expr}/{testee}_{btype}_{expr}_matches.csv")



In [29]:
res = get_basic_metrics(scores)

+----------+---------------+------------------+--------------+----------------+------------+
| Judge    | Testee        | Experiment       | Q-Type       |   Data Samples |   Accuracy |
+==========+===============+==================+==============+================+============+
| qwen3-4B | gpt-4.1-mini  | baseline         | Qualitative  |            106 |   0.188679 |
+----------+---------------+------------------+--------------+----------------+------------+
| qwen3-4B | gpt-4.1-mini  | baseline         | Quantitative |             92 |   0.423913 |
+----------+---------------+------------------+--------------+----------------+------------+
| qwen3-4B | qwen2.5-7B-it | baseline         | Qualitative  |            106 |   0.235849 |
+----------+---------------+------------------+--------------+----------------+------------+
| qwen3-4B | qwen2.5-7B-it | baseline         | Quantitative |             92 |   0.130435 |
+----------+---------------+------------------+--------------+--------

In [34]:
comparisons = []
pairs = [("baseline", "strategic"), ("baseline", "forward"), ("forward", "strategic")]
btypes = ["qual", "quant"]
testees = ["gpt", "qwen"]
bench = "gpqa"

for p1, p2 in pairs:
    for testee in testees:
        for btype in btypes:
            comparisons.append((f"scores/{bench}/{p1}/{testee}_{btype}_{p1}_matches.csv",
                                f"scores/{bench}/{p2}/{testee}_{btype}_{p2}_matches.csv"))
            
results_c = get_comparisons(comparisons)

+----------+---------------+-----------------------------+--------------+------------------+-------------------+-------------+-------------+------------+-------------+----------------+-----------------+------------------+-------------------+
| Judge    | Testee        | Experiment                  | Q-Type       |   Decision Flips |   Decision Flip % |     p-value |   Cohen's d |   Base ASR |   Gamed ASR |   Base Samples |   Gamed Samples |   Base Successes |   Gamed Successes |
+==========+===============+=============================+==============+==================+===================+=============+=============+============+=============+================+=================+==================+===================+
| qwen3-4B | gpt-4.1-mini  | baseline/strategic/         | Qualitative  |               10 |         0.0943396 | 0.49815     |   0.0927089 |   0.188679 |    0.226415 |            106 |             106 |               20 |                24 |
+----------+---------------+----

In [32]:
#mean_Accuracies
def get_avg_acc_over_bench(qual_scores):
    means = []
    judge = "qwen3-4B"
    
    for score in qual_scores:
        df1 = pd.read_csv(score)
        df2 = pd.read_csv(score.replace("qual", "quant"))
        mean_acc, tot = mean_accuracy(pd.concat([df1, df2], ignore_index=True))
        testee, exp, qtype = get_exp(score)
        means.append([judge, testee, exp, tot, mean_acc])

    headers = ["Judge", "Testee", "Experiment", "Data Samples", "Accuracy"]
    table = tabulate(means, headers=headers, tablefmt="grid")
    print(table)

    df = pd.DataFrame(means, columns=["Judge", "Testee","Experiment", "Data Samples", "Accuracy"])
    fig = px.bar(
        df,
        x="Experiment",
        y="Accuracy",
        color="Testee",   # distinct color for each Model + Q-Type
        barmode="group",
        facet_col="Judge",
        text="Accuracy",
        title=f"Average Accuracies - GPQA-D",
        color_discrete_map={
        "gpt-4.1-mini": "#8c564b",
        "qwen2.5-7B-it": "#70ad47"
        }

    )
    

    fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
    fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide', width=600,
        height=500)
    fig.show()

qual_scores = [s for s in scores if "qual" in s]
get_avg_acc_over_bench(qual_scores)



+----------+---------------+------------------+----------------+------------+
| Judge    | Testee        | Experiment       |   Data Samples |   Accuracy |
+==========+===============+==================+================+============+
| qwen3-4B | gpt-4.1-mini  | baseline         |            198 |   0.29798  |
+----------+---------------+------------------+----------------+------------+
| qwen3-4B | qwen2.5-7B-it | baseline         |            198 |   0.186869 |
+----------+---------------+------------------+----------------+------------+
| qwen3-4B | gpt-4.1-mini  | strategic        |            198 |   0.207071 |
+----------+---------------+------------------+----------------+------------+
| qwen3-4B | qwen2.5-7B-it | strategic        |            198 |   0.141414 |
+----------+---------------+------------------+----------------+------------+
| qwen3-4B | gpt-4.1-mini  | multiple-forward |            198 |   0.19697  |
+----------+---------------+------------------+----------------+

In [35]:

df = pd.DataFrame(res, columns=["Judge", "Testee","Experiment", "Q-Type", "DataPoints", "Accuracy"])
df["Model_Qtype"] = df["Testee"] + " (" + df["Q-Type"] + ")"

fig = px.bar(
    df,
    x="Experiment",
    y="Accuracy",
    color="Model_Qtype",   # distinct color for each Model + Q-Type
    barmode="group",
    facet_col="Judge",
    text="Accuracy",
    title="Accuracy per Experiment grouped by Judge, Testee, and Q-Type"
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
fig.show()

